# C.2 最後答對代表過程正確嗎？


題目是「2+2=?」，回答卻寫「2+2=5；5−1=4，所以答案4」。最後數字確實是4，但第一條等式錯，後面還多減了一次1。前置是[步驟與最後答案的區別](https://birdhackor.github.io/tiny-perceptron-vlm/C.1.html)。這份回答不能因結尾正確就被當作可靠解法；我們要分開看每條等式、步驟接續與最後答案。

先把每步限制為可直接檢查的加法三元組 `(a,b,claimed)`，代表「a+b應等於claimed」。減1寫成加-1，避免另外實作減法。第一步是 `(2,2,5)`，第二步是 `(5,-1,4)`。逐步檢查只問每條等式是否算對；接續檢查則問第二步是否使用上一個claimed當起點。最後再把最後claimed和題目真值4比較，三種判準分別記錄。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
steps = [(2, 2, 5), (5, -1, 4)]
truth = 2 + 2
valid = [a + b == claimed for a, b, claimed in steps]
linked = [steps[i][0] == steps[i - 1][2] for i in range(1, len(steps))]
print("等式算對", valid)
print("接上前步", linked)
print("最後答案對", steps[-1][2] == truth)

輸出應依序是 `[False,True]`、`[True]`、True。第一步2+2不等於5；第二步5加-1確實是4，也確實從上一個聲稱的5開始。這說明接得上不代表前提正確，結尾正確也不能補救最早的算術錯誤。`range(1,len(steps))`從第二步開始逐項比對，因第一步沒有上一個claimed；最後的-1索引表示清單最後一筆。

如果只把第一步claimed由5改4，兩條等式各自都算對，卻變成「2+2=4；5−1=4」：第二步仍從5開始，接續檢查會False。反過來，即使等式和接續都通過，也不一定符合題目要求，例如無理由地加減額外數值仍需要檢查任務意義。本例的正確解法其實只需2+2=4，不能因驗證器會查加法，就假定它能判斷任何文字論證。

這類程式稱verifier（驗證器），它依明確規則檢查指定範圍的輸出。對結構化算術可以做精確核對；對自然語言、開放事實或較長證明，規則覆蓋可能不足，也可能需要人工判斷。檢查結果應說清楚驗了什麼，例如「等式正確」與「解答任務正確」，不能只留一個模糊的通過標籤。

練習只把第一步最後的5改成4，保持第二步不變。先預測等式變 `[True,True]`、接續變 `[False]`、最後答案仍True，再執行核對。請指出第二步哪個數字沿用了舊版本，並說明為何刪掉不必要的第二步，比修出一條更長的補救敘述更直接。
